# 08 - MinIO vs SeaweedFS (Experiment 13)

## What you'll learn
- Run the same Iceberg workload on two S3-compatible backends side by side.
- Compare object layout, commit cost, and LIST cost after a small-file storm.
- Read the same table with PyIceberg and DuckDB on each backend.

## Interview questions this answers
- How do you choose object storage for a lakehouse?
- What S3 semantics does Iceberg depend on?
- Why does the small-file problem stress object storage, not just query engines?

In [ ]:
import os
import time
from datetime import datetime, timezone

import duckdb
import pandas as pd
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import DoubleType, LongType, NestedField, StringType, TimestampType

from src.catalog_helper import (
    StorageBackend,
    configure_duckdb_for_s3,
    ensure_namespace,
    get_catalog,
    get_s3_client,
    list_object_keys,
    storage_console_url,
)

## Step 1 — Health check both backends

In [ ]:
BACKENDS: list[StorageBackend] = ["minio", "seaweed"]

for backend in BACKENDS:
    bucket = os.environ.get(
        "MINIO_BUCKET" if backend == "minio" else "SEAWEED_BUCKET",
        "warehouse",
    )
    buckets = [b["Name"] for b in get_s3_client(backend).list_buckets()["Buckets"]]
    assert bucket in buckets, f"{backend}: missing bucket {bucket}"
    catalog = get_catalog(backend)
    ensure_namespace(catalog)
    print(f"{backend}: bucket={bucket} console={storage_console_url(backend)} catalog OK")

## Step 2 — Mirror table: same schema, three appends

In [ ]:
SCHEMA = Schema(
    NestedField(1, "event_id", StringType(), required=True),
    NestedField(2, "event_ts", TimestampType(), required=True),
    NestedField(3, "user_id", LongType()),
    NestedField(4, "amount", DoubleType()),
)
TABLE_ID = "lab.storage_compare"


def sample_batch(batch_id: int, rows: int = 10) -> pa.Table:
    now = datetime.now(timezone.utc)
    return pa.table(
        {
            "event_id": [f"{batch_id}-{i}" for i in range(rows)],
            "event_ts": [now] * rows,
            "user_id": list(range(rows)),
            "amount": [float(i) for i in range(rows)],
        }
    )


tables = {}
for backend in BACKENDS:
    catalog = get_catalog(backend)
    ensure_namespace(catalog)
    try:
        catalog.drop_table(TABLE_ID)
    except Exception:
        pass
    table = catalog.create_table(TABLE_ID, schema=SCHEMA)
    for batch_id in range(3):
        table.append(sample_batch(batch_id))
    tables[backend] = catalog.load_table(TABLE_ID)
    print(backend, "location:", tables[backend].location())

## Step 3 — Object layout under metadata/ and data/

In [ ]:
layout_rows = []
for backend in BACKENDS:
    loc = tables[backend].location()
    # s3://warehouse/lab/storage_compare
    _, prefix = loc.replace("s3://", "").split("/", 1)
    prefix = prefix.rstrip("/") + "/"
    keys = list_object_keys(prefix, backend=backend)
    meta = [k for k in keys if "/metadata/" in k]
    data = [k for k in keys if "/data/" in k]
    layout_rows.append(
        {
            "backend": backend,
            "total_objects": len(keys),
            "metadata_objects": len(meta),
            "data_objects": len(data),
        }
    )

pd.DataFrame(layout_rows)

## Step 4 — PyIceberg and DuckDB reads

In [ ]:
for backend in BACKENDS:
    table = tables[backend]
    py_rows = table.scan().to_arrow().num_rows
    con = duckdb.connect()
    configure_duckdb_for_s3(con, backend)
    duck_rows = con.execute(
        f"SELECT count(*) FROM iceberg_scan('{table.location()}')"
    ).fetchone()[0]
    print(f"{backend}: pyiceberg={py_rows} duckdb={duck_rows}")

## Step 5 — Small-file storm (200 single-row commits)

In [ ]:
STORM_COMMITS = 200
STORM_TABLE_ID = "lab.storage_storm"


def run_small_file_storm(backend: StorageBackend) -> dict:
    catalog = get_catalog(backend)
    ensure_namespace(catalog)
    try:
        catalog.drop_table(STORM_TABLE_ID)
    except Exception:
        pass
    table = catalog.create_table(STORM_TABLE_ID, schema=SCHEMA)

    t0 = time.perf_counter()
    for i in range(STORM_COMMITS):
        table.append(sample_batch(i, rows=1))
    write_sec = time.perf_counter() - t0

    table = catalog.load_table(STORM_TABLE_ID)
    prefix = table.location().replace("s3://", "").split("/", 1)[1].rstrip("/") + "/"

    t1 = time.perf_counter()
    keys = list_object_keys(prefix, backend=backend)
    list_sec = time.perf_counter() - t1

    data_files = len([k for k in keys if "/data/" in k and k.endswith(".parquet")])
    return {
        "backend": backend,
        "commits": STORM_COMMITS,
        "data_files": data_files,
        "write_sec": round(write_sec, 2),
        "list_sec": round(list_sec, 4),
        "total_objects": len(keys),
    }


storm_results = [run_small_file_storm(b) for b in BACKENDS]
pd.DataFrame(storm_results)

## Try yourself
- Open MinIO console and SeaweedFS master UI; browse `warehouse/lab/storage_compare/`.
- Continue with **Break it** steps in `experiments/13-minio-vs-seaweedfs.md`.
- Compare `catalog.db` vs `catalog_seaweed.db` with `sqlite3` from the `lab/` directory.